# ICU Length of Stay Prediction

In [5]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import polars as pl
from datetime import timedelta
import numpy as np

from ethos.constants import PROJECT_ROOT
from ethos.constants import SpecialToken as ST
from ethos.inference.constants import Reason, Task
from ethos.metrics import compute_and_print_metrics_regression, preprocess_inference_results
from ethos.utils import get_mimic_hf_patients, get_mimic_sepsis_icu_stays

# Requires original MIMIC-IV 2.2 dataset that we do not provide
mimic_sepsis = get_mimic_sepsis_icu_stays()
# heart_failure_cohort = get_mimic_hf_patients(PROJECT_ROOT / "data/mimic-2.2")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# DEBUGGING ONLY
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / 'pyproject.toml').exists() and project_root != project_root.parent:
    project_root = project_root.parent

sys.path.insert(0, str(project_root))
from src.ethos.datasets.length_of_stay import LosPredictionDataset
path = "../../../../mnt/data_share/project_henri/ethos-ares/mimic-tokenized/test"
dataset = LosPredictionDataset(path, offset=timedelta(hours=24))

# MIMIC

In [6]:
results_dir = PROJECT_ROOT / "results"
los_pred_result_dir = results_dir / Task.LOS_PREDICTION

def preprocess_los_prediction_results(input_dir) -> pl.DataFrame:
    return preprocess_inference_results(
        input_dir,
        actual_expr=pl.col("actual") == ST.ICU_DISCHARGE,
        expected_expr=pl.col("expected") == ST.ICU_DISCHARGE,
        filter_ambiguous=(
            pl.col("actual").is_in([ST.DEATH, ST.ICU_DISCHARGE])
            & (pl.col("stop_reason") == Reason.GOT_TOKEN)
        ),
    )

In [9]:
los_pred_results = []

for result_fp in los_pred_result_dir.iterdir():
    df = preprocess_los_prediction_results(result_fp)
    y_pred = [y.total_seconds() for y in df["true_token_time"]]
    y_true = [y.total_seconds() for y in df["token_time"]]

    res_full = compute_and_print_metrics_regression(np.array(y_true), np.array(y_pred))
    mae_td = timedelta(seconds=res_full["mae"])
    ci_lower_td = timedelta(seconds=res_full["ci"][0])
    ci_upper_td = timedelta(seconds=res_full["ci"][1])
    print(f"MAE: {mae_td} for {len(df)*df['counts'][0]} samples")
    print(f"95% CI: {ci_lower_td} - {ci_upper_td}")

2026-10-28 17:08:33.769 | WARNING  | ethos.metrics:preprocess_inference_results:326 - Dropped 2,582 (1.17%) ambiguous results.


MAE: 3 days, 6:41:33.891739 for 219940 samples
95% CI: 3 days, 4:06:45.788882 - 3 days, 9:21:32.257172
